In [1]:
import pandas as pd
import io
import requests
import numpy as np
import datetime
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

# Data

In [2]:
def getDF(league,season):
    "Function to pull data from website football-data.co.uk"
    #Inputs
    #   League: 2 letter string with country and division reference (I1 for Italian 1st division, I2 for Italian Second Division ,... [EPL is referenced as E0])
    #   Season: string to reference the season to get (0405 for the 2004/2005 season)
    #Returns pandas dataframe of football-data data for inputted league and season
    rooturl="https://www.football-data.co.uk/mmz4281/"
    url= rooturl+str(season)+'/'+league+'.csv'
    s0=requests.get(url).content
    df=pd.read_csv(io.StringIO(s0.decode('utf-8')))
    #Handle Renaming of Odds Columns for Over/Under 2.5 Odds from 19/20 onwards
    RENAME = {
    "BbAv>2.5": "Avg>2.5",
    "BbAv<2.5": "Avg<2.5",
    "BbMx>2.5": "Max>2.5",
    "BbMx<2.5": "Max<2.5",
    }
    df=df.rename(RENAME, axis=1)
    KEEP = ["Div","Date","HomeTeam","AwayTeam","FTHG","FTAG","FTR","HS", "AS","Avg>2.5","Avg<2.5"]
    df=df[KEEP].reindex(columns=KEEP)
    df['Date']=pd.to_datetime(df["Date"], dayfirst=True, format='mixed')
    return df

def year2Str(season:int)->str:
    "Function to convert integer season to 4 character string for season in order to pull data from football-data"
    #Input: season as an integer value (506 for the 05/06 season)
    #Returns season as a string ('05/06' in the case of 506 being the input)
    return str(season).zfill(4)

In [3]:
#Seasons we want to pull data for are 05/06-25/26
seasons=np.arange(506,2527,101)
seasons=list(map(year2Str,seasons))
#We will pull data from the  5 major leagues
leagues=['D1','E0','F1','I1','SP1']
#seasonDict stores the dataframes in a dictionary, key being the league
leagueDict={}
#Populating seasonDict with nested for loop
for l in leagues:
    leagueDict[l]={}
    seasonList=[]
    for s in seasons:
        sdf=getDF(l,s)
        sdf.insert(0,'Season',s)
        seasonList.append(sdf)
    leagueDict[l]=pd.concat(seasonList)
    leagueDict[l]=leagueDict[l].reset_index(drop=True)
    leagueDict[l].insert(0,"match_id",l+'_'+leagueDict[l].index.astype(str))


In [4]:
leagueDict[l]

,match_id,Season,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,Avg>2.5,Avg<2.5
0,SP1_0,0506,SP1,2005-08-27,Alaves,Barcelona,0,0,D,5,17,1.78,1.92
1,SP1_1,0506,SP1,2005-08-27,Ath Bilbao,Sociedad,3,0,H,10,9,1.85,1.84
2,SP1_2,0506,SP1,2005-08-27,Valencia,Betis,1,0,H,9,14,1.99,1.72
3,SP1_3,0506,SP1,2005-08-28,Ath Madrid,Zaragoza,0,0,D,16,9,1.97,1.74
4,SP1_4,0506,SP1,2005-08-28,Cadiz,Real Madrid,1,2,A,15,17,1.70,1.98
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7975,SP1_7975,2526,SP1,2026-05-23,Girona,Elche,1,1,D,10,3,1.65,2.14
7976,SP1_7976,2526,SP1,2026-05-23,Mallorca,Oviedo,3,0,H,23,7,1.83,1.90
7977,SP1_7977,2526,SP1,2026-05-23,Real Madrid,Ath Bilbao,4,2,H,13,8,1.47,2.57
7978,SP1_7978,2526,SP1,2026-05-23,Valencia,Barcelona,3,1,H,19,11,1.43,2.68


In [5]:
#Check dates have been processed correctly
#All dates should fall between August and May
for l in leagueDict.keys():
    print(l)
    print(leagueDict[l].groupby("Season")["Date"].agg(["min","max"]))

D1
              min        max
Season                      
0506   2005-08-05 2006-05-13
0607   2006-08-11 2007-05-19
0708   2007-08-10 2008-05-17
0809   2008-08-15 2009-05-23
0910   2009-08-07 2010-05-08
1011   2010-08-20 2011-05-14
1112   2011-08-05 2012-05-05
1213   2012-08-24 2013-05-18
1314   2013-08-09 2014-05-10
1415   2014-08-22 2015-05-23
1516   2015-08-14 2016-05-14
1617   2016-08-26 2017-05-20
1718   2017-08-18 2018-05-12
1819   2018-08-24 2019-05-18
1920   2019-08-16 2020-06-27
2021   2020-09-18 2021-05-22
2122   2021-08-13 2022-05-14
2223   2022-08-05 2023-05-27
2324   2023-08-18 2024-05-18
2425   2024-08-23 2025-05-17
2526   2025-08-22 2026-05-16
E0
              min        max
Season                      
0506   2005-08-13 2006-05-07
0607   2006-08-19 2007-05-13
0708   2007-08-11 2008-05-11
0809   2008-08-16 2009-05-24
0910   2009-08-15 2010-05-09
1011   2010-08-14 2011-05-22
1112   2011-08-13 2012-05-13
1213   2012-08-18 2013-05-19
1314   2013-08-17 2014-05-11
1415   2

# Feature Engineering



In [ ]:
def formFeatures(matches:pd.DataFrame, window=5, tranCols=["FTHG","FTAG","HS","AS"],statsNames=["GF","GA","SF","SA"]) ->pd.DataFrame:
    "Function to build form features from imported DataFrame. The DataFrame long contains 1 row per team per match to calculate the rolling form statistics."
    #Inputs: matches->DataFrame with columns ['Div', 'Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HS', 'AS', 'Avg>2.5', 'Avg<2.5', 'Season'] 
    #       window->Number of matches in the rolling window
    #       tranCols->Names of columns to be transformed (MUST BE IN ORDERED TEAM PAIRS FOR EACH STATISTIC)
    #       statsNames-> Names of transformed columns (MUST BE IN ORDERED TEAM PAIRS FOR EACH STATISTIC, WITH CORRESPONDING VALUES FOR tranCols)
    #Output: Same DataFrame with added columns: ['HL{window}GS', 'AL{window}GS','HL{window}GC', 'AL{window}GC'...]
    #       HL{n},AL{n} -> Home Last n matches, Away Last n matches
    #       GS,GC -> Goals Scored, Goals Conceded
    #       SF,SA -> Shots For, Shots Against
    assert len(tranCols) == len(statsNames), "column and stat lists must be same length"
    assert len(tranCols) % 2 == 0, "columns must be in for/against pairs"
    def homeAwayMapper(tranCols=tranCols,statsNames=statsNames):
        "Function that maps stats to home and away"
        #Inputs: tranCols -> Columns in original DataFrame to be transformed. 
        #       statsNames -> Columns in output dataframe
        '''Input lists must be in the format where columns and stats are grouped in with their inverse (i.e. same stat for opposite team).
        For example: The lists ['FTHG','FTAG','HS','AS'], ["GF","GA","SF","SA"] works, the lists ['FTHG','HS','FTAG','AS'], ["GF","GA","SF","SA"] will not map properly
        '''
        #Outputs: Ordered mappings for each column and stat to use for the home and away tables.
        homeMap={'HomeTeam':'Team', 'AwayTeam':'Opponent'}
        awayMap={'HomeTeam':'Opponent', 'AwayTeam':'Team'}
        for h, a, fo, ag in zip(tranCols[0::2], tranCols[1::2],statsNames[0::2], statsNames[1::2]):
            homeMap.update({h:fo,a:ag})
            awayMap.update({a:fo,h:ag})
        return homeMap, awayMap
    homeMap,awayMap=homeAwayMapper()
    home = matches[["match_id","Date","Season","HomeTeam","AwayTeam","FTR"]+list(tranCols)].rename(columns=homeMap)
    home["venue"] = "H"
    away = matches[["match_id","Date","Season","HomeTeam","AwayTeam","FTR"]+list(tranCols)].rename(columns=awayMap)
    away["venue"] = "A"
    long=pd.concat([home,away], ignore_index=True).sort_values('Date') 
    def prior_mean(s, n=window):
        "Mean over the previous n matches, excluding the current one."
        #Input: s->Column swith numeric values, n-> number of previous values for mean to be taken over:
        #Output: Column where each cell has the mean of n previous values (first n values are na).
        return s.shift(1).rolling(n).mean()
    g = long.groupby("Team")

    long[["L"+str(window) + c for c in statsNames]] = g[statsNames].transform(prior_mean)
    longH = long[long.venue=="H"].rename({f"L{window}{s}": f"HL{window}{s}" for s in statsNames}, axis=1)
    longA = long[long.venue=="A"].rename({f"L{window}{s}": f"AL{window}{s}" for s in statsNames}, axis=1)

    matches=matches.merge(longH[["match_id"]+["HL"+str(window) + c for c in statsNames]], on='match_id', how='left')
    return matches.merge(longA[["match_id"]+["AL"+str(window) + c for c in statsNames]], on='match_id', how='left')





In [67]:
#Check formFeatures works as intendes
assert(len(formFeatures(leagueDict['D1']))==len(leagueDict['D1'])) 

aSet=leagueDict['D1'].head(100)
aFormSet=formFeatures(aSet)
checkSet=aFormSet.loc[(aFormSet['HomeTeam']=='Bayern Munich')|(aFormSet['AwayTeam']=='Bayern Munich')|(aFormSet['HomeTeam']=='Ein Frankfurt')|(aFormSet['AwayTeam']=='Ein Frankfurt')].head(11)
print(checkSet.columns)

ValueError: Cannot subset columns with a tuple with more than one element. Use a list instead.

In [ ]:
def homeAwayMapper(tranCols=["FTHG","FTAG","HS","AS"],statsNames=["GF","GA","SF","SA"]):
    "Function that maps stats to home and away"
    #Inputs: tranCols -> Columns in original DataFrame to be transformed. 
    #       statsNames -> Columns in output dataframe
    '''Input lists must be in the format where columns and stats are grouped in with their inverse (i.e. same stat for opposite team).
    For example: The lists ['FTHG','FTAG','HS','AS'], ["GF","GA","SF","SA"] works, the lists ['FTHG','HS','FTAG','AS'], ["GF","GA","SF","SA"] will not map properly
    '''
    #Outputs: Ordered mappings for each column and stat to use for the home and away tables.
    homeMap={'HomeTeam':'Team', 'AwayTeam':'Opponent'}
    awayMap={'HomeTeam':'Opponent', 'AwayTeam':'Team'}
    for h, a, fo, ag in zip(tranCols[0::2], tranCols[1::2],statsNames[0::2], statsNames[1::2]):
        homeMap.update({h:fo,a:ag})
        awayMap.update({a:fo,h:ag})
    return homeMap, awayMap
print(homeAwayMapper())

({'HomeTeam': 'Team', 'AwayTeam': 'Opponent', 'FTHG': 'GF', 'FTAG': 'GA', 'HS': 'SF', 'AS': 'SA'}, {'HomeTeam': 'Opponent', 'AwayTeam': 'Team', 'FTAG': 'GF', 'FTHG': 'GA', 'AS': 'SF', 'HS': 'SA'})
